# 正交化与 QR

先手算两列的新方向，再用数值库核对分解与最小二乘。QR 的符号选择可以不同，只要重建与正交关系成立。

依赖：NumPy、Matplotlib。按顺序运行，先在纸上预测再核对；图内使用英文标签。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
def check_prediction(value, expected, label):
    if value is None:
        print(label + '：尚未填写，请先在纸上计算，再替换 None。')
        return None
    try:
        proposed = np.asarray(value, dtype=float)
    except (TypeError, ValueError, OverflowError):
        print(label + '：请填写数值、数值列表或数组。')
        return False
    reference = np.asarray(expected, dtype=float)
    if proposed.shape != reference.shape:
        print(label + '：尺寸应为 ' + str(reference.shape) + '，请检查行列。')
        return False
    if not np.all(np.isfinite(proposed)):
        print(label + '：请检查是否除以零或产生了非有限数。')
        return False
    correct = bool(np.allclose(proposed, reference))
    print(label + ('：核对正确。' if correct else '：尚不一致，请回到方程或内积检查。'))
    return correct


## 先写下第二列的剩余方向

a1=(1,1,0)、a2=(1,0,1)。先归一化a1，计算a2沿q1的投影，再从a2减去它。把剩余向量填入 `predicted_remaining`；后面的原实验将逐项核对。

预测区不调用QR，也不以读者答案作为后续计算输入。


In [ ]:
predicted_remaining = None
check_prediction(predicted_remaining, [1/2,-1/2,1], '减去投影后的方向')


## 减掉重叠方向

a2 在 q1 方向上有多少？减掉以后是否垂直？


In [ ]:
A = np.array([[1.,1.],[1.,0.],[0.,1.]])
q1 = A[:,0]/np.linalg.norm(A[:,0])
remaining = A[:,1]-q1*(q1 @ A[:,1])
q2 = remaining/np.linalg.norm(remaining)
Qhand = np.column_stack((q1,q2))
Rhand = Qhand.T @ A
print('剩余方向:',remaining,'手算 R:',Rhand)
assert np.allclose(Qhand.T @ Qhand,np.eye(2))
assert np.allclose(Qhand @ Rhand,A)


## 正交坐标与回代

b=(1,2,2)，先求 Q^Tb，再从最后一行求解。


In [ ]:
Q,R = np.linalg.qr(A,mode='reduced')
b = np.array([1.,2.,2.])
x = np.linalg.solve(R,Q.T @ b)
residual = b-A @ x
print('系数:',x,'预测:',A @ x,'残差:',residual)
assert np.allclose(x,[1,1])
assert np.allclose(A.T @ residual,0)
assert np.allclose(Q @ R,A)
assert np.allclose(Q.T @ Q,np.eye(2))


## 独立方向越来越小

把第二列设为第一列加 epsilon*(0,0,1)。先预测剩余长度。这里只观察教学公式，不把它当生产求解器。


In [ ]:
for epsilon in [1.,1e-3,1e-8,0.]:
    first = np.array([1.,1.,0.])
    second = first+epsilon*np.array([0.,0.,1.])
    q = first/np.linalg.norm(first)
    remainder = second-q*(q @ second)
    print('epsilon:',epsilon,'剩余长度:',np.linalg.norm(remainder))


## 明确手算回代，而不是只调用 solve

用手工 Qhand、Rhand 与 b=(1,2,2)。先求 Qhand^T b，再从最后一行求 x2，代到第一行求 x1。这里Qhand三行两列，不能写Qhand的普通逆。


In [ ]:
b_fit = np.array([1.,2.,2.])
hand_coordinates = Qhand.T @ b_fit
x2_hand = hand_coordinates[1] / Rhand[1,1]
x1_hand = (hand_coordinates[0]-Rhand[0,1]*x2_hand)/Rhand[0,0]
hand_solution = np.array([x1_hand,x2_hand])
print('orthogonal coordinates:', hand_coordinates, 'back substitution:', hand_solution)
assert np.allclose(hand_solution, [1,1])
assert np.allclose(np.tril(Rhand,-1), 0)
assert np.allclose(Rhand @ hand_solution, hand_coordinates)
assert np.allclose(A.T @ (b_fit-A @ hand_solution), 0)
Pfit = Qhand @ Qhand.T
for trial in [np.array([0.,0.]), np.array([2.,-1.])]:
    error2 = np.linalg.norm(b_fit-A @ trial)**2
    outside2 = np.linalg.norm((np.eye(3)-Pfit) @ b_fit)**2
    inside2 = np.linalg.norm(hand_coordinates-Rhand @ trial)**2
    assert np.isclose(error2, outside2+inside2)


<details><summary>核对</summary>
第二列减去的投影是 (1/2,1/2,0)，剩余为 (1/2,-1/2,1)。两列在新基下的坐标组成上三角R。回代得到(1,1)；最优误差来自列空间外，系数不能消除它。改变Q的一支符号时，对应R的行也必须一起变号。
</details>


## 合上实验，自己解释

证明原列与新正交基能互相表示，说明R为什么上三角。先用手工Q取观测坐标，再回代，不写长方形Q的普通逆。最后把第二列设成第一列本身，解释哪一步不能继续，以及浮点残余为什么不一定恰好显示为零。
